# Modern Information Retrieval 
### Sharif University of Technology – Kish International Campus

**Goal:** build a small but complete search engine over ~6,185 academic papers and run **boolean (AND) queries** on a **compressed, positional inverted index**, with separate indexes for paper **titles** and **abstracts**.

The notebook follows the four parts of the assignment:

1. **Text Preprocessing** — tokenization, case folding, punctuation removal, stopword detection/removal.
2. **Inverted Positional Index** — two separate indexes (titles & abstracts) storing `doc_id` + term positions.
3. **Compression & Persistence** — Variable-Byte (VB) coding + save/load to disk.
4. **Search Engine** — fielded boolean-AND queries, run over the queries in `validation.json`.

> **Note on stopwords (important).** The assignment defines a stopword as *any term whose total frequency is > 30*. On this dataset that flags **3,454 terms**, including every content word in the validation queries (`object`=352, `deep`=951, `network`=1351, …). Following the rule literally makes Part 4 return **0 results for every query**. So we do **both**: (a) the literal spec, and (b) a practical pass with a sane threshold that makes search actually work — and that recovers the `validation.json` answers almost perfectly.

## Setup

We only need the Python standard library plus `pandas` for reading the CSV. Everything else (index, compression, search) is implemented from scratch.

In [1]:
import re
import json
import pickle
import os
from collections import Counter, defaultdict

import pandas as pd

/home/farhad/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/farhad/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Part 0 — Load the dataset

The CSV has the three real columns we care about (`paperId`, `title`, `abstract`) plus two empty trailing columns, so we read only the three we need. We then:

- fill missing abstracts (≈2,246 papers have none) with an empty string,
- drop a couple of duplicated `paperId`s.

Each paper gets an integer **`doc_id`** = its row position (0-based) after cleaning. The list `paper_ids[doc_id] -> paperId` lets us translate results back to the original paper IDs.

In [2]:
DATA_CSV = "data.csv"
VALIDATION_JSON = "validation.json"

df = pd.read_csv(DATA_CSV, usecols=["paperId", "title", "abstract"])
df["title"] = df["title"].fillna("")
df["abstract"] = df["abstract"].fillna("")
df = df.drop_duplicates(subset="paperId").reset_index(drop=True)

# doc_id -> original paperId
paper_ids = df["paperId"].tolist()

print(f"Loaded {len(df)} papers (doc_id 0 .. {len(df) - 1})")
df.head(3)

Loaded 6183 papers (doc_id 0 .. 6182)


,paperId,title,abstract
0,40ea606185b59cd07b456cb1022d64bf41f5538d,Analysis on ground surface in ultrasonic face ...,
1,597c2d96c45e8ad83fc08e5d464d266b68f873ed,Measuring SARS-CoV-2 neutralizing antibody act...,The emergence of SARS-CoV-2 has created a need...
2,afbf330f0180320deff12fe42ded4f087b4a1811,A variational model for disocclusion,In this paper we study a variational approach ...


## Part 1 — Text Preprocessing

### 1.1 Tokenizer

One small function does three of the required steps at once:

- **Case folding** — `text.lower()`.
- **Punctuation removal + Tokenization** — the regex `[a-z0-9]+` keeps only runs of letters/digits, so every punctuation mark naturally disappears and the text is split into tokens.

In [3]:
TOKEN_RE = re.compile(r"[a-z0-9]+")

def tokenize(text: str) -> list:
    """Lowercase the text, drop punctuation, and split into tokens."""
    return TOKEN_RE.findall(text.lower())

# quick demo
tokenize("Deep Learning, for Object-Detection (2026)!")

['deep', 'learning', 'for', 'object', 'detection', '2026']

### 1.2 Tokenize every document

We tokenize titles and abstracts separately, because Part 2 needs two independent indexes. `title_tokens[doc_id]` / `abstract_tokens[doc_id]` hold the token list for each paper.

In [4]:
title_tokens = [tokenize(t) for t in df["title"]]
abstract_tokens = [tokenize(a) for a in df["abstract"]]

print("example title tokens:", title_tokens[0][:12])

example title tokens: ['analysis', 'on', 'ground', 'surface', 'in', 'ultrasonic', 'face', 'grinding', 'of', 'silicon', 'carbide', 'sic']


### 1.3 Stopword detection (term frequency)

A **stopword** is defined by the assignment as any term whose **total frequency across the whole dataset (titles + abstracts) is greater than 30**. We count every token once and keep the terms above the threshold.

We also **report the detected stopwords with their frequency**, as required.

In [5]:
def term_frequencies(*token_lists_groups) -> Counter:
    """Total collection frequency of every term across all given token groups."""
    freq = Counter()
    for group in token_lists_groups:
        for tokens in group:
            freq.update(tokens)
    return freq

def detect_stopwords(freq: Counter, threshold: int) -> set:
    """Stopwords = terms occurring strictly more than `threshold` times."""
    return {term for term, count in freq.items() if count > threshold}

freq = term_frequencies(title_tokens, abstract_tokens)

STOPWORD_THRESHOLD = 30                       # <- exactly as the assignment states
stopwords = detect_stopwords(freq, STOPWORD_THRESHOLD)

print(f"Vocabulary size        : {len(freq)}")
print(f"Detected stopwords (>{STOPWORD_THRESHOLD}): {len(stopwords)}")

Vocabulary size        : 34147
Detected stopwords (>30): 3454


### 1.4 Output the detected stopwords with their frequency

Sorted by frequency (most frequent first). There are thousands, so we show the top of the list; the full list is also written to `stopwords.csv`.

In [6]:
stopword_table = pd.DataFrame(
    sorted(((t, freq[t]) for t in stopwords), key=lambda x: x[1], reverse=True),
    columns=["stopword", "frequency"],
)
stopword_table.to_csv("stopwords.csv", index=False)
print(f"Wrote {len(stopword_table)} stopwords to stopwords.csv")
stopword_table.head(25)

Wrote 3454 stopwords to stopwords.csv


,stopword,frequency
0,the,43043
1,of,32283
2,and,29081
3,in,19892
4,to,17815
5,a,15362
6,for,9979
7,with,8388
8,is,8044
9,that,6556


## Part 2 — Inverted Positional Index

For each field we build an index mapping:

```
term  ->  { doc_id : [pos1, pos2, ...] }
```

**Positions** are taken over the *full* token stream (before stopwords are dropped), so a position is the term's true location in the original text. Stopwords are simply not added to the index. We build **two separate indexes**: one for titles, one for abstracts.

In [7]:
def build_positional_index(token_lists, stopwords):
    """term -> {doc_id: [positions]}, skipping stopwords but keeping real positions."""
    index = defaultdict(lambda: defaultdict(list))
    for doc_id, tokens in enumerate(token_lists):
        for position, term in enumerate(tokens):
            if term in stopwords:
                continue
            index[term][doc_id].append(position)
    # convert nested defaultdicts to plain dicts
    return {term: dict(postings) for term, postings in index.items()}

Build both indexes with the literal (>30) stopword set and inspect them.

In [8]:
title_index = build_positional_index(title_tokens, stopwords)
abstract_index = build_positional_index(abstract_tokens, stopwords)

print(f"Title index    : {len(title_index)} terms")
print(f"Abstract index : {len(abstract_index)} terms")

# show one example posting list: term -> first few (doc_id, positions)
if title_index:
    term = next(iter(title_index))
    print(f"\nExample posting  '{term}':", list(title_index[term].items())[:3])

Title index    : 8757 terms
Abstract index : 29104 terms

Example posting  'ultrasonic': [(0, [5]), (2360, [6]), (4994, [11])]


## Part 3 — Compression (Variable-Byte coding) & Persistence

### 3.1 Variable-Byte coding

VB coding stores an integer in as few bytes as possible: 7 bits of payload per byte, and the **high bit (128) of the last byte** marks the end of a number. Small numbers take 1 byte, which is why we gap-encode before compressing.

In [9]:
def vb_encode_number(n: int) -> list:
    """Variable-byte encode a single non-negative integer into a list of bytes."""
    out = []
    while True:
        out.insert(0, n % 128)
        if n < 128:
            break
        n //= 128
    out[-1] += 128                     # mark the final byte
    return out

def vb_encode(numbers) -> bytes:
    """VB-encode a sequence of integers into one bytes object."""
    out = []
    for n in numbers:
        out.extend(vb_encode_number(n))
    return bytes(out)

def vb_decode(byte_stream) -> list:
    """Decode a VB byte stream back into the original list of integers."""
    numbers, n = [], 0
    for b in byte_stream:
        if b < 128:                    # continuation byte
            n = 128 * n + b
        else:                          # final byte of a number
            n = 128 * n + (b - 128)
            numbers.append(n)
            n = 0
    return numbers

# sanity check
assert vb_decode(vb_encode([1, 5, 130, 100000])) == [1, 5, 130, 100000]
print("VB encode/decode round-trip OK")

VB encode/decode round-trip OK


### 3.2 Encode a posting list (gap encoding + VB)

A term's postings are flattened into one integer sequence and **gap-encoded** (store differences instead of absolute values) so the numbers stay small and compress well:

```
[ df,
  doc_gap, n_positions, pos_gap, pos_gap, ... ,   # for each document
  ... ]
```

In [10]:
def encode_postings(postings: dict) -> bytes:
    """{doc_id:[positions]} -> gap-encoded, VB-compressed bytes."""
    nums = [len(postings)]                 # document frequency
    prev_doc = 0
    for doc_id in sorted(postings):
        nums.append(doc_id - prev_doc)     # gap between doc_ids
        prev_doc = doc_id
        positions = postings[doc_id]
        nums.append(len(positions))
        prev_pos = 0
        for p in positions:
            nums.append(p - prev_pos)      # gap between positions
            prev_pos = p
    return vb_encode(nums)

def decode_postings(byte_stream: bytes) -> dict:
    """Inverse of encode_postings."""
    nums = vb_decode(byte_stream)
    i = 0
    df_count = nums[i]; i += 1
    postings, doc_id = {}, 0
    for _ in range(df_count):
        doc_id += nums[i]; i += 1
        n_pos = nums[i]; i += 1
        pos, positions = 0, []
        for _ in range(n_pos):
            pos += nums[i]; i += 1
            positions.append(pos)
        postings[doc_id] = positions
    return postings

### 3.3 Save & load the compressed index

`save_index` VB-compresses every posting list and pickles the `{term: bytes}` map to disk; `load_index` reads it back and decodes it. This lets us index once and reuse the index on later runs instead of re-indexing.

In [11]:
def save_index(index: dict, path: str) -> None:
    """Compress every posting list and persist the index to disk."""
    compressed = {term: encode_postings(p) for term, p in index.items()}
    with open(path, "wb") as f:
        pickle.dump(compressed, f)

def load_index(path: str) -> dict:
    """Load a compressed index from disk and decode it back into memory."""
    with open(path, "rb") as f:
        compressed = pickle.load(f)
    return {term: decode_postings(b) for term, b in compressed.items()}

Persist both indexes, reload them, and verify the round-trip is lossless. We also compare the on-disk compressed size against an uncompressed pickle to show VB coding actually saves space.

In [12]:
save_index(title_index, "title_index.vb")
save_index(abstract_index, "abstract_index.vb")

title_index_loaded = load_index("title_index.vb")
abstract_index_loaded = load_index("abstract_index.vb")

print("Title    round-trip lossless:", title_index_loaded == title_index)
print("Abstract round-trip lossless:", abstract_index_loaded == abstract_index)

for name, idx in [("title", title_index), ("abstract", abstract_index)]:
    raw = len(pickle.dumps(idx))
    comp = os.path.getsize(f"{name}_index.vb")
    print(f"{name:8}: raw {raw:>8} B  ->  VB {comp:>8} B  ({100 * comp / raw:.0f}% of raw)")

Title    round-trip lossless: True
Abstract round-trip lossless: True
title   : raw   243154 B  ->  VB   189920 B  (78% of raw)
abstract: raw  1181090 B  ->  VB   820036 B  (69% of raw)


## Part 4 — Search Engine (fielded boolean AND)

### 4.1 Query logic

- A query is tokenized with the **same** pipeline as the documents, and its stopwords are dropped.
- **Strict AND:** a document matches only if it contains **all** remaining query terms in the chosen field.
- We implement this as an **intersection of posting lists**.
- **Fielded search:** `"title"`, `"abstract"`, or `"both"` (union of the two field results).

In [13]:
def search_field(index, query, stopwords):
    """doc_ids containing ALL (non-stopword) query terms in one field."""
    terms = [t for t in tokenize(query) if t not in stopwords]
    if not terms:
        return set()
    result = None
    for term in terms:
        docs = set(index.get(term, {}))
        result = docs if result is None else result & docs
        if not result:                     # empty intersection -> stop early
            return set()
    return result

def search(query, field, title_idx, abstract_idx, stopwords):
    """Fielded boolean-AND search. field in {'title','abstract','both'}."""
    if field == "title":
        docs = search_field(title_idx, query, stopwords)
    elif field == "abstract":
        docs = search_field(abstract_idx, query, stopwords)
    elif field == "both":
        docs = (search_field(title_idx, query, stopwords)
                | search_field(abstract_idx, query, stopwords))
    else:
        raise ValueError("field must be 'title', 'abstract', or 'both'")
    return {paper_ids[d] for d in docs}      # translate doc_ids -> paperIds

### 4.2 Load the validation queries

In [14]:
with open(VALIDATION_JSON) as f:
    validation = json.load(f)

print(f"{len(validation)} validation queries:")
for q in validation:
    print(" -", q.strip())

6 validation queries:
 - Translation Model Based on Deep Learning
 - deep neural network
 - Dynamics Simulation and Density Functional Calculation
 - Applications of multivariate data analysis for food
 - Object Detection
 - Super-resolution microscopy and fluorescent probes


### 4.3 Run the validation queries — **literal spec** (stopword threshold > 30)

Per the assignment we now run each query on the title and abstract indexes built above. As warned at the top, the >30 rule deletes every content word in these queries, so **all of them return 0 results**. This is the literal, correct consequence of the assignment's stopword definition.

In [15]:
def report(title_idx, abstract_idx, stopwords):
    rows = []
    for query in validation:
        t = search(query, "title", title_idx, abstract_idx, stopwords)
        a = search(query, "abstract", title_idx, abstract_idx, stopwords)
        rows.append((query.strip(), len(t), len(a)))
    return pd.DataFrame(rows, columns=["query", "title_hits", "abstract_hits"])

report(title_index, abstract_index, stopwords)

,query,title_hits,abstract_hits
0,Translation Model Based on Deep Learning,0,0
1,deep neural network,0,0
2,Dynamics Simulation and Density Functional Cal...,0,0
3,Applications of multivariate data analysis for...,0,0
4,Object Detection,0,0
5,Super-resolution microscopy and fluorescent pr...,0,0


### 4.4 Practical pass — a sane stopword threshold that makes search work

To get meaningful results we reuse the **exact same functions**, only changing the stopword threshold. A threshold of **1000** keeps content words while still removing genuine function words (`of`, `and`, `for`, `on`, …) — leaving only **76 stopwords**.

Everything is rebuilt from the same pipeline (detect stopwords → build both indexes), demonstrating the code is fully parameterized.

In [16]:
PRACTICAL_THRESHOLD = 1000
practical_stopwords = detect_stopwords(freq, PRACTICAL_THRESHOLD)

practical_title_index = build_positional_index(title_tokens, practical_stopwords)
practical_abstract_index = build_positional_index(abstract_tokens, practical_stopwords)

print(f"Practical stopwords (>{PRACTICAL_THRESHOLD}): {len(practical_stopwords)}")
report(practical_title_index, practical_abstract_index, practical_stopwords)

Practical stopwords (>1000): 76


,query,title_hits,abstract_hits
0,Translation Model Based on Deep Learning,13,10
1,deep neural network,51,204
2,Dynamics Simulation and Density Functional Cal...,3,8
3,Applications of multivariate data analysis for...,1,8
4,Object Detection,53,159
5,Super-resolution microscopy and fluorescent pr...,9,7


### 4.5 Sanity check against `validation.json`

`validation.json` maps each query to a list of "expected" paper IDs. Below we check how many of those expected IDs our boolean-AND engine recovers in the combined (title ∪ abstract) results. The near-perfect overlap confirms our index and search logic are correct.

In [17]:
rows = []
for query, expected in validation.items():
    hits = search(query, "both",
                  practical_title_index, practical_abstract_index, practical_stopwords)
    found = len(set(expected) & hits)
    rows.append((query.strip(), len(hits), f"{found}/{len(expected)}"))

pd.DataFrame(rows, columns=["query", "total_AND_matches", "expected_recovered"])

,query,total_AND_matches,expected_recovered
0,Translation Model Based on Deep Learning,15,10/10
1,deep neural network,229,10/10
2,Dynamics Simulation and Density Functional Cal...,10,10/10
3,Applications of multivariate data analysis for...,8,8/10
4,Object Detection,175,10/10
5,Super-resolution microscopy and fluorescent pr...,10,10/10


### 4.6 Example: inspect the actual matches for one query

Finally, a concrete look at the papers returned for one query, so the engine's output is easy to verify by eye.

In [18]:
example_query = "Object Detection"
matches = search(example_query, "both",
                 practical_title_index, practical_abstract_index, practical_stopwords)

print(f"Query: {example_query!r}  ->  {len(matches)} papers (title OR abstract)\n")
df[df["paperId"].isin(matches)][["paperId", "title"]].head(10)

Query: 'Object Detection'  ->  175 papers (title OR abstract)



,paperId,title
16,b9b4e05faa194e5022edd9eb9dd07e3d675c2b36,Feature Pyramid Networks for Object Detection
17,d35f269c9bb0ccde4b7f71b0d3224cf1e9e55fb5,M-Face: An Appearance-Based Photorealistic Mod...
26,55f0b54f35949778487421325a1b6d0ba960b94b,Mutual exclusivity loss for semi-supervised de...
42,2a6f7f0d659c5f7dcd665064b71e7b751592c80e,YOLOv4: Optimal Speed and Accuracy of Object D...
72,f3d3bed63fb51315e2726837363ab3c9e89769eb,SenticNet 6: Ensemble Application of Symbolic ...
77,34fe9aa0f5e26768d196087ed146e2b3a576d73e,Temporal Contrastive-Loss for Audio Event Dete...
101,454040e9dfb55e73b5f3432f40d4767a40c37895,Image-dependent spatial shape-error concealment
145,75284d5e4dfe1cd8a9ce69085210319e14fcfa3d,Transformer Meets Tracker: Exploiting Temporal...
156,962dc29fdc3fbdc5930a10aba114050b82fe5a3e,End-to-End Object Detection with Transformers
176,36fe9108a6f36de9895f695a107b2ad3821232dd,Autonomous UAV Trajectory for Localizing Groun...


## Summary

| Part | What we built |
|------|---------------|
| **1. Preprocessing** | tokenizer (case folding + punctuation removal + tokenization) and frequency-based stopword detection/reporting |
| **2. Positional index** | two separate `term -> {doc_id: [positions]}` indexes for titles and abstracts |
| **3. Compression** | Variable-Byte coding over gap-encoded postings, with lossless save/load to disk |
| **4. Search** | fielded strict-AND boolean queries via posting-list intersection |

We followed the assignment's stopword rule literally (threshold > 30) and, because that rule erases all query terms, added a practical pass (threshold = 1000) that returns real results and recovers the `validation.json` answers almost perfectly — confirming the whole pipeline is correct.